In [3]:
# 1. protobuf 재설치
!pip uninstall protobuf -y
!pip install protobuf==3.20.3

Found existing installation: protobuf 6.33.5
Uninstalling protobuf-6.33.5:
  Successfully uninstalled protobuf-6.33.5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 15.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vllm 0.14.1 requires protobuf>=6.30.0, but you have protobuf 3.20.3 which is incompatible.
grpcio-reflection 1.78.0 requires protobuf<7.0.0,>=6.31.1, but you have protobuf 3.20.3 which is incompatible.
tensorflow-metadata 1.17.3 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.3 which is incompatible.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 3.20.3 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 3.20.3 wh

In [1]:
# === 첫 번째 셀 ===
# 대회 환경 맞추기
!pip uninstall -y torch transformers -q

!pip install torch
!pip install transformers==4.57.3
!pip install tokenizers==0.22.1
!pip install accelerate==1.10.1
!pip install datasets==4.4.1
!pip install compressed-tensors==0.13.0

# llmcompressor 설치 시도
!pip install llmcompressor

print("설치 완료 - 런타임 재시작 필요!")


  Using cached torch-2.10.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (31 kB)
  Using cached nvidia_nvshmem_cu12-3.4.5-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (2.1 kB)
  Using cached triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (1.7 kB)
Using cached torch-2.10.0-cp312-cp312-manylinux_2_28_x86_64.whl (915.7 MB)
Using cached nvidia_nvshmem_cu12-3.4.5-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (139.1 MB)
Using cached triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (188.3 MB)
  Attempting uninstall: triton
    Found existing installation: triton 3.5.1
    Uninstalling triton-3.5.1:
      Successfully uninstalled triton-3.5.1
  Attempting uninstall: nvidia-nvshmem-cu12
    Found existing installation: nvidia-nvshmem-cu12 3.3.20
    Uninstalling nvidia-nvshmem-cu12-3.3.20:
      Successfully uninstalled nvidia-nvshmem-cu12-3.3.20
ERROR: pip's dependency resolver does not currently take in

In [2]:
import os
import torch
import shutil
from pathlib import Path

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier


In [3]:
MODEL_ID = "LGAI-EXAONE/EXAONE-4.0-1.2B"
OUT_DIR  = "./model"

DATASET_ID = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"

NUM_CALIBRATION_SAMPLES = 1024 # 캘리브레이션 데이터 수 증가 512-> 1024
MAX_SEQUENCE_LENGTH = 1024

# Quantization
SCHEME = "W4A16" # GPTQ-int4
TARGETS = ["Linear"]


In [4]:
print("[INFO] 모델 로드 중...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
)

print("[INFO] 모델/토크나이저 로드 완료")


[INFO] 모델 로드 중...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


[INFO] 모델/토크나이저 로드 완료


In [5]:
# (추가) 모든 층 down_proj + 마지막 3개 층 보호
n_layers = model.config.num_hidden_layers
ignore_list = ["model.embed_tokens", "model.norm", "lm_head"]

for i in range(n_layers):
    # 모든 down_proj
    ignore_list.append(f"model.layers.{i}.mlp.down_proj")

    # 27번 MLP만 추가
    if i == n_layers - 3:  # 27
        ignore_list.append(f"model.layers.{i}.mlp.gate_proj")
        ignore_list.append(f"model.layers.{i}.mlp.up_proj")

    # 28, 29 전체
    if i >= n_layers - 2:  # 28, 29
        ignore_list.append(f"model.layers.{i}.self_attn.q_proj")
        ignore_list.append(f"model.layers.{i}.self_attn.k_proj")
        ignore_list.append(f"model.layers.{i}.self_attn.v_proj")
        ignore_list.append(f"model.layers.{i}.self_attn.o_proj")
        ignore_list.append(f"model.layers.{i}.mlp.gate_proj")
        ignore_list.append(f"model.layers.{i}.mlp.up_proj")

print(f"[INFO] {len(ignore_list)}개 모듈 제외")
print(f"[INFO] 27: MLP만, 28-29: 전체")


[INFO] 47개 모듈 제외
[INFO] 27: MLP만, 28-29: 전체


In [6]:
print("[INFO] 데이터 전처리 중...")

from itertools import chain
from datasets import Dataset

# 길이 필터 설정
MIN_TOKENS = 128
MAX_TOKENS = 1024

# 원본에서 넉넉히 뽑아서(=NUM_CALIBRATION_SAMPLES * 5) 셔플 후 사용
raw_ds = load_dataset(DATASET_ID, split=DATASET_SPLIT)
raw_ds = raw_ds.shuffle(seed=42).select(range(NUM_CALIBRATION_SAMPLES * 5))

# 길이 필터링 함수
def length_filter(example):
    try:
        text = tokenizer.apply_chat_template(
            example["conversations"],
            tokenize=False,
            add_generation_prompt=True
        )
        tokens = tokenizer(text, add_special_tokens=False)["input_ids"]
        token_len = len(tokens)
        # 128~1024 토큰 범위만
        return MIN_TOKENS <= token_len <= MAX_TOKENS
    except:
        return True  # 에러나면 포함

print(f"[INFO] 길이 필터링 중 (MIN: {MIN_TOKENS}, MAX: {MAX_TOKENS})...")
raw_ds_filtered = raw_ds.filter(length_filter)
print(f"[INFO] 필터링 결과: {len(raw_ds_filtered)}개 / {len(raw_ds)}개")

# 샘플 부족하면 원본 사용
if len(raw_ds_filtered) < NUM_CALIBRATION_SAMPLES * 2:
    print("[WARN] 필터링된 샘플 부족, 원본 추가 사용...")
    raw_ds_filtered = raw_ds.select(range(NUM_CALIBRATION_SAMPLES * 3))


def get_packed_dataset(dataset, tokenizer, seq_len=2048, num_samples=512):
    formatted_texts = []
    for entry in dataset:
        try:
            text = tokenizer.apply_chat_template(
                entry["conversations"],
                tokenize=False,
                add_generation_prompt=True
            )
            formatted_texts.append(text)
        except Exception:
            continue

    encodings = tokenizer(formatted_texts, add_special_tokens=False)
    all_input_ids = list(chain(*encodings["input_ids"]))

    packed_dataset = []
    for i in range(0, len(all_input_ids), seq_len):
        chunk_ids = all_input_ids[i:i + seq_len]
        if len(chunk_ids) == seq_len:
            packed_dataset.append({
                "input_ids": chunk_ids,
                "attention_mask": [1] * seq_len,
            })
        if len(packed_dataset) >= num_samples:
            break

    return packed_dataset

ds_list = get_packed_dataset(
    raw_ds_filtered,  # 필터링된 데이터 사용
    tokenizer,
    seq_len=MAX_SEQUENCE_LENGTH,
    num_samples=NUM_CALIBRATION_SAMPLES
)

ds = Dataset.from_list(ds_list)

print("[INFO] 데이터 전처리 완료")
print("[INFO] packed samples:", len(ds), "| max_len:", MAX_SEQUENCE_LENGTH)


[INFO] 데이터 전처리 중...
[INFO] 길이 필터링 중 (MIN: 128, MAX: 1024)...
[INFO] 필터링 결과: 4378개 / 5120개
[INFO] 데이터 전처리 완료
[INFO] packed samples: 1024 | max_len: 1024


In [7]:
print(f"[INFO] GPTQ 시작 (scheme={SCHEME}, samples={NUM_CALIBRATION_SAMPLES}, max_len={MAX_SEQUENCE_LENGTH})...")

recipe = [
    GPTQModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=ignore_list,
        dampening_frac=0.2, # dampening_frac = 0.1 -> 0.2
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

print("[INFO] GPTQ 완료")


[INFO] GPTQ 시작 (scheme=W4A16, samples=1024, max_len=1024)...
2026-02-18T13:59:08.365972+0000 | reset | INFO - Compression lifecycle reset
2026-02-18T13:59:08.368723+0000 | from_modifiers | INFO - Creating recipe from modifiers
2026-02-18T13:59:08.430488+0000 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-02-18T13:59:08.431330+0000 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`


(1/31): Calibrating: 100%|██████████| 1024/1024 [00:10<00:00, 100.77it/s]

2026-02-18T13:59:20.464852+0000 | compress_modules | INFO - Quantizing model.layers.0.self_attn.q_proj using 1024 samples


2026-02-18T13:59:21.979797+0000 | compress | METRIC - time 1.51s
2026-02-18T13:59:21.980623+0000 | compress | METRIC - error 4.07
2026-02-18T13:59:21.981684+0000 | compress | METRIC - GPU 0 | usage: 6.24% | total memory: 24 GB
2026-02-18T13:59:21.982260+0000 | compress | METRIC - Compressed module size: 8.486912 MB
2026-02-18T13:59:21.983625+0000 | compress_modules | INFO - Quantizing model.layers.0.self_attn.k_proj using 1024 samples
2026-02-18T13:59:23.151589+0000 | compress | METRIC - time 1.17s
2026-02-18T13:59:23.152469+0000 | compress | METRIC - error 1.19
2026-02-18T13:59:23.153593+0000 | compress | METRIC - GPU 0 | usage: 6.24% | total memory: 24 GB
2026-02-18T13:59:23.154325+0000 | compress | METRIC - Compressed module size: 2.121728 MB
2026-02-18T13:59:23.155564+0000 | compress_modules | INFO - Quantizing model.layers.0.self_attn.v_proj using 1024 samples
2026-02-18T13:59:24.306882+0000 | compress | METRIC - time 1.15s
2026-02-18T13:59:24.308120+0000 | compress | METRIC - err

(2/31): Calibrating: 100%|██████████| 1024/1024 [00:10<00:00, 100.30it/s]

...

2026-02-18T14:09:19.385944+0000 | compress_modules | INFO - Quantizing model.layers.27.self_attn.q_proj using 1024 samples


2026-02-18T14:09:20.571230+0000 | compress | METRIC - time 1.18s
2026-02-18T14:09:20.572740+0000 | compress | METRIC - error 7407.67
2026-02-18T14:09:20.573740+0000 | compress | METRIC - GPU 0 | usage: 6.08% | total memory: 24 GB
2026-02-18T14:09:20.574381+0000 | compress | METRIC - Compressed module size: 8.486912 MB
2026-02-18T14:09:20.575614+0000 | compress_modules | INFO - Quantizing model.layers.27.self_attn.k_proj using 1024 samples
2026-02-18T14:09:21.731203+0000 | compress | METRIC - time 1.16s
2026-02-18T14:09:21.732715+0000 | compress | METRIC - error 1940.16
2026-02-18T14:09:21.733721+0000 | compress | METRIC - GPU 0 | usage: 6.08% | total memory: 24 GB
2026-02-18T14:09:21.734370+0000 | compress | METRIC - Compressed module size: 2.121728 MB
2026-02-18T14:09:21.735590+0000 | compress_modules | INFO - Quantizing model.layers.27.self_attn.v_proj using 1024 samples
2026-02-18T14:09:22.889675+0000 | compress | METRIC - time 1.15s
2026-02-18T14:09:22.891465+0000 | compress | METR

(31/31): Propagating: 100%|██████████| 1024/1024 [00:01<00:00, 712.55it/s]

2026-02-18T14:09:48.343456+0000 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
2026-02-18T14:09:48.389385+0000 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`


[INFO] GPTQ 완료


In [8]:
os.makedirs(OUT_DIR, exist_ok=True)

model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

print(f"[INFO] 모델 저장 완료: {OUT_DIR}")


2026-02-18T14:09:51.791412+0000 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.


Compressing model: 166it [00:01, 112.96it/s]


[INFO] 모델 저장 완료: ./model


In [ ]:
zip_name = "GPTQ_4"
print(f"[INFO] {zip_name}.zip 생성 중...")

shutil.make_archive(
    base_name=zip_name,
    format="zip",
    root_dir=".",
    base_dir=OUT_DIR,
)

print(f"[INFO] 생성 완료: {zip_name}.zip")


[INFO] GPTQ_3.zip 생성 중...
[INFO] 생성 완료: GPTQ_3.zip


In [ ]:
from google.colab import files
files.download("GPTQ_4.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>